# Exploratory Data Analysis — COVID-19 Brazil

# 1. Imports

In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt
from pandas import DataFrame
from IPython.display import Markdown, display

# 2. Data Preparation

This notebook is self-contained: the cleaning pipeline is re-applied here rather than depending on `data_quality.ipynb` having been run first. This guarantees reproducibility — the EDA can be executed independently at any time.

## 2.1 Load Raw Data

CSV has no native null type. Values recorded as `"NA"`, `""`, or `"Importados/Indefinidos"` are read by pandas as plain strings — invisible to `isna()` and unaffected by interpolation. Replacing them with `numpy.nan` at load time makes them proper missing values from the start.

In [ ]:
dataset = pandas.read_csv("data/caso_full.csv")

dataset = dataset.replace(
    ["<NA>", "NA", "NaN", "nan", "null", "", "Importados/Indefinidos"],
    numpy.nan
)

display(Markdown(f"**Rows loaded:** {len(dataset):,}"))

## 2.2 Cleaning Pipeline

Four decisions are applied in sequence. Order matters — each step depends on the previous one.

**`place_type == "state"`** — The dataset contains both city-level and state-level rows for the same dates. State rows already aggregate all cities within them, so keeping both granularities would double-count every case. Since the forecasting model operates at state level, the EDA must use the same granularity.

**`is_last == True`** — The dataset preserves the full history of secretariat corrections: the same state + date combination appears multiple times as numbers are revised over time. Only the most recent revision is the authoritative value. Using any other version means analyzing stale data.

**`sort_values(["state", "date"])`** — Interpolation estimates missing values from neighboring rows. Those neighbors must be the chronologically adjacent days for the same state. Sorting by state then date guarantees this — without it, interpolation would draw from arbitrary rows.

**`where(~group["is_repeated"])`** — `is_repeated = True` means the secretariat published no bulletin that day, so `new_confirmed` is forced to `0` by convention — it does not mean zero new cases. The `where` call turns those zeros into `NaN`, making them eligible for interpolation. Keeping the `0` would teach the model that cases periodically vanish.

**`interpolate(method="linear")`** — Linear interpolation estimates each missing day from its neighbors. Linear is appropriate for daily data with short gaps (typically 1–2 days on weekends). Forward fill would reproduce the same zero-on-no-publication behavior being corrected; spline or polynomial would overfit single-day gaps.

**`.clip(lower=0)`** — Negative values arise when a secretariat reclassifies cases between municipalities. The model cannot learn from a negative case count. Clipping to zero is the least harmful treatment: it does not drop the row (which would create a gap) or interpolate (the value is known, just physically impossible).

In [ ]:
df = dataset[
    (dataset["place_type"] == "state") &
    (dataset["is_last"] == True)
].copy().sort_values(["state", "date"])

def clean_state_series(group):
    group = group.copy()
    group["new_confirmed"] = (
        group["new_confirmed"]
        .where(~group["is_repeated"])
        .interpolate(method="linear")
        .clip(lower=0)
    )
    return group

df = pandas.concat([
    clean_state_series(group)
    for _, group in df.groupby("state")
])

df["date"] = pandas.to_datetime(df["date"])

display(Markdown(f"**Records after cleaning:** {len(df):,}"))
display(Markdown(f"**Date range:** {df['date'].min().date()} \u2192 {df['date'].max().date()}"))
display(Markdown(f"**States:** {df['state'].nunique()}"))
df.head()

# 3. National Time Series

Aggregates `new_confirmed` across all states to show the full arc of the pandemic in Brazil. The 7-day rolling average smooths out the weekly reporting noise (secretariats often publish fewer cases on weekends and catch up on Mondays), making the epidemic waves clearly visible.

In [ ]:
national = (
    df.groupby("date")["new_confirmed"]
    .sum()
    .sort_index()
)

rolling_7d = national.rolling(7, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 4))

ax.fill_between(national.index, national.values, color="#90caf9", alpha=0.4, label="Daily")
ax.plot(national.index, rolling_7d, color="#1565c0", lw=2, label="7-day rolling average")

ax.set_title("New Confirmed Cases — Brazil (all states aggregated)", fontsize=13, pad=12)
ax.set_xlabel("Date")
ax.set_ylabel("New Cases")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

display(Markdown(f"**Peak (daily):** {national.max():,.0f} cases on {national.idxmax().date()}"))

# 4. State Comparison

Small multiples for the 9 states with the highest total case counts. Each subplot uses an independent y-axis (`sharey=False`) because absolute volumes differ by orders of magnitude between SP and smaller states — a shared axis would flatten every series except São Paulo.

In [ ]:
top_states = (
    df.groupby("state")["new_confirmed"]
    .sum()
    .nlargest(9)
    .index.tolist()
)

fig, axes = plt.subplots(3, 3, figsize=(16, 10), sharey=False)

for ax, state in zip(axes.flat, top_states):
    s = df[df["state"] == state].set_index("date")["new_confirmed"]
    roll = s.rolling(7, center=True).mean()
    ax.fill_between(s.index, s.values, color="#90caf9", alpha=0.4, label="Daily")
    ax.plot(s.index, roll, color="#1565c0", lw=1.5, label="7-day avg")
    ax.set_title(state, fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1_000:.0f}k" if x >= 1000 else str(int(x)))
    )
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("New Confirmed Cases — Top 9 States (7-day rolling average)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4.1 Regional Comparison: Nordeste vs Sudeste

Aggregates cases by region to compare epidemic dynamics. The two regions have different population sizes, so the series are normalized per 100k inhabitants to make the comparison fair — otherwise Sudeste would dominate purely due to scale.

In [ ]:
NORDESTE = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
SUDESTE  = ["ES", "MG", "RJ", "SP"]

region_pop = {
    "Nordeste": 57_071_654,
    "Sudeste":  88_371_433,
}

region_series = {}
for label, states in [("Nordeste", NORDESTE), ("Sudeste", SUDESTE)]:
    total = (
        df[df["state"].isin(states)]
        .groupby("date")["new_confirmed"]
        .sum()
        .sort_index()
    )
    region_series[label] = total / region_pop[label] * 100_000

colors = {"Nordeste": "#e05c5c", "Sudeste": "#1565c0"}

fig, ax = plt.subplots(figsize=(14, 5))

for label, series in region_series.items():
    ax.plot(
        series.index,
        series.rolling(7, center=True).mean(),
        color=colors[label],
        lw=2,
        label=label,
    )
    ax.fill_between(series.index, series.rolling(7, center=True).mean(), alpha=0.08, color=colors[label])

ax.set_title("New Confirmed Cases per 100k Inhabitants — Nordeste vs Sudeste", fontsize=13, pad=12)
ax.set_xlabel("Date")
ax.set_ylabel("Cases per 100k inhabitants")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

for label, series in region_series.items():
    peak_val  = series.max()
    peak_date = series.idxmax().date()
    display(Markdown(f"**{label} peak:** {peak_val:.1f} cases/100k on {peak_date}"))